# Embedding Validation
This notebook tests the `FeatureProcessor` from `src/embedder.py` (specifically embeddings and column dropping) on the cleaned dataset.

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Add src to path
sys.path.append('../')
from src.embedder import FeatureProcessor

processor = FeatureProcessor()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8576.29it/s]


In [2]:
# Load data
data_path = "../data/processed/company_metrics_clean.parquet"

df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} rows")
display(df.head())

Loaded 6045 rows


,ticker,forwardPE,ev_to_ebitda,ebitda_margin,sector,industry,business_summary,ebitda,total_cash,total_debt,shares_outstanding,debt_to_ebitda
0,NVDA,17.739650,35.888,0.61698,Technology,Semiconductors,NVIDIA Corporation operates as a data center s...,1.332300e+11,6.255600e+10,1.141200e+10,2.430000e+10,0.085656
1,GOOGL,25.076876,26.757,0.37279,Communication Services,Internet Content & Information,Alphabet Inc. offers various products and plat...,1.501750e+11,1.268430e+11,6.699600e+10,5.822000e+09,0.446119
2,AAPL,28.615034,25.736,0.35100,Technology,Consumer Electronics,"Apple Inc. designs, manufactures, and markets ...",1.529020e+11,6.690700e+10,9.050900e+10,1.468114e+10,0.591941
3,MSFT,21.752918,17.616,0.57377,Technology,Software - Infrastructure,Microsoft Corporation develops and supports so...,1.752590e+11,8.946200e+10,1.232780e+11,7.425629e+09,0.703405
4,AMZN,26.442839,18.686,0.20327,Consumer Cyclical,Internet Retail,"Amazon.com, Inc. engages in the retail sale of...",1.457310e+11,1.230290e+11,1.785470e+11,1.075425e+10,1.225182


## 1. Test Embedding
Using `all-MiniLM-L6-v2` local model.

In [7]:
df_emb = processor.embed_summaries(df.head(10)) # Small sample

display(df_emb.iloc[:,12:396].head())

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.59it/s]


,nlp_0,nlp_1,nlp_2,nlp_3,nlp_4,nlp_5,nlp_6,nlp_7,nlp_8,nlp_9,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,-0.031164,-0.069812,-0.048435,-0.014469,-0.036502,-0.090951,-0.013756,0.004428,0.024300,-0.041028,...,-0.052249,-0.020048,0.002744,-0.099040,0.015237,0.016853,-0.034988,-0.053926,0.036351,0.030003
1,-0.059735,-0.116992,0.044080,-0.085275,-0.003429,0.052656,-0.020798,-0.021409,0.073483,-0.009831,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,-0.042323,-0.021482,-0.009602,-0.061581,0.073229,0.068119,0.091094,-0.049078,0.058306,-0.022786,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,-0.014270,-0.056247,0.000315,-0.068629,0.030435,0.009555,0.075610,-0.029651,0.043202,0.026177,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,0.039800,-0.067963,-0.031519,-0.043795,0.089798,0.047744,-0.004417,-0.015469,0.088615,-0.003279,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between first two companies
sim = cosine_similarity([df_emb.loc[0,"nlp_0":"nlp_383"]], [df_emb.loc[1,"nlp_0":"nlp_383"]])
print(f"Similarity between {df_emb.iloc[0]['ticker']} and {df_emb.iloc[1]['ticker']}: {sim[0][0]:.4f}")

Similarity between NVDA and GOOGL: 0.4206


## 2. Test Column Dropping
Ensuring that non-numeric columns are removed for ML readiness.

In [9]:
# Demonstrate dropping extra columns
df_ml = processor.drop_extra_columns(df_emb)
print(f"Columns after dropping: {df_ml.columns.tolist()[:10]}...")
print(f"Remaining columns: {len(df_ml.columns)}")

display(df_ml.head())

Failed to find embeddings


Columns after dropping: ['forwardPE', 'ev_to_ebitda', 'ebitda_margin', 'ebitda', 'total_cash', 'total_debt', 'shares_outstanding', 'debt_to_ebitda', 'nlp_0', 'nlp_1']...
Remaining columns: 392


,forwardPE,ev_to_ebitda,ebitda_margin,ebitda,total_cash,total_debt,shares_outstanding,debt_to_ebitda,nlp_0,nlp_1,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,17.739650,35.888,0.61698,1.332300e+11,6.255600e+10,1.141200e+10,2.430000e+10,0.085656,-0.031164,-0.069812,...,-0.052249,-0.020048,0.002744,-0.099040,0.015237,0.016853,-0.034988,-0.053926,0.036351,0.030003
1,25.076876,26.757,0.37279,1.501750e+11,1.268430e+11,6.699600e+10,5.822000e+09,0.446119,-0.059735,-0.116992,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,28.615034,25.736,0.35100,1.529020e+11,6.690700e+10,9.050900e+10,1.468114e+10,0.591941,-0.042323,-0.021482,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,21.752918,17.616,0.57377,1.752590e+11,8.946200e+10,1.232780e+11,7.425629e+09,0.703405,-0.014270,-0.056247,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,26.442839,18.686,0.20327,1.457310e+11,1.230290e+11,1.785470e+11,1.075425e+10,1.225182,0.039800,-0.067963,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085


## 3. Verify Final ML Dataset
Loading the final processed file to verify its structure.

In [10]:
data_path = "../data/processed/company_metrics_ml.parquet"
df = pd.read_parquet(data_path)

df


,forwardPE,ev_to_ebitda,ebitda_margin,ebitda,total_cash,total_debt,shares_outstanding,debt_to_ebitda,nlp_0,nlp_1,...,nlp_374,nlp_375,nlp_376,nlp_377,nlp_378,nlp_379,nlp_380,nlp_381,nlp_382,nlp_383
0,17.739650,35.888,0.61698,1.332300e+11,6.255600e+10,1.141200e+10,2.430000e+10,0.085656,-0.031164,-0.069812,...,-0.052249,-0.020048,0.002744,-0.099040,0.015237,0.016853,-0.034988,-0.053926,0.036351,0.030003
1,25.076876,26.757,0.37279,1.501750e+11,1.268430e+11,6.699600e+10,5.822000e+09,0.446119,-0.059735,-0.116992,...,0.011315,-0.044257,-0.024711,-0.135265,0.032019,0.070319,-0.015535,-0.074552,0.032307,-0.036426
2,28.615034,25.736,0.35100,1.529020e+11,6.690700e+10,9.050900e+10,1.468114e+10,0.591941,-0.042323,-0.021482,...,-0.040436,0.024960,0.037074,-0.097955,0.022558,0.120378,0.046419,-0.064816,0.104943,0.013485
3,21.752918,17.616,0.57377,1.752590e+11,8.946200e+10,1.232780e+11,7.425629e+09,0.703405,-0.014270,-0.056247,...,0.022693,0.035224,0.073339,-0.102653,0.079596,0.118757,-0.010114,-0.027585,0.031214,-0.044478
4,26.442839,18.686,0.20327,1.457310e+11,1.230290e+11,1.785470e+11,1.075425e+10,1.225182,0.039800,-0.067963,...,0.040109,-0.000710,0.027882,-0.100945,0.072389,0.122982,-0.006967,-0.052227,0.004188,0.016085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6040,14.022167,0.000,-0.23284,-1.674758e+06,0.000000e+00,0.000000e+00,7.996261e+09,-0.000000,-0.056173,-0.048288,...,-0.050170,-0.101823,0.022734,-0.054379,-0.053292,-0.043625,0.019533,-0.064665,0.003089,0.058151
6041,-9.400000,-1.879,-0.03605,-9.222400e+07,3.088200e+07,1.238270e+08,2.519140e+07,-1.342677,-0.014797,0.045850,...,0.050779,0.002931,0.045317,-0.066792,0.106199,0.052362,-0.052725,-0.072798,0.072390,0.042886
6042,-1.325444,0.000,-0.03686,-2.322380e+05,0.000000e+00,0.000000e+00,1.687964e+07,-0.000000,-0.055846,-0.040478,...,-0.067763,0.058794,0.023222,-0.045942,-0.021594,0.084824,0.030711,-0.013880,0.010142,0.027954
6043,-8.285714,-0.198,0.00000,-1.864100e+07,1.107000e+06,2.500000e+04,2.676637e+06,-0.001341,-0.016013,-0.075610,...,-0.070225,0.087178,-0.057583,-0.065913,0.090664,-0.085205,0.040502,-0.094684,0.057274,0.028732
